In [ ]:
#| default_exp web_pages

# Pages

> Complete page components for common web application patterns

In [ ]:
#| export
import importlib



from fh_matui.components import *
from fh_matui.core import *
from fasthtml.common import *
from fh_matui.page_snippets import *
from fh_matui.foundations import *

from fastcore.utils import partial
from fasthtml.common import *
from fasthtml.jupyter import FastHTML, fast_app, JupyUvi, HTMX
from fastlite import *
import fasthtml.components as fc
from fasthtml.common import A, Button as FhButton, I, Span
import socket
import time
import subprocess

In [ ]:
#| code-fold: true
#| eval: false

def kill_process_on_port(port):
    """Kill any process using the specified port on Windows"""
    try:
        result = subprocess.run(
            f'netstat -ano | findstr :{port}',
            shell=True, capture_output=True, text=True
        )
        
        if result.stdout:
            lines = result.stdout.strip().split('\n')
            for line in lines:
                if 'LISTENING' in line:
                    pid = line.strip().split()[-1]
                    subprocess.run(f'taskkill /PID {pid} /F', shell=True, capture_output=True)
                    print(f"✓ Killed process {pid} on port {port}")
                    time.sleep(0.5)
                    return True
        return False
    except Exception as e:
        print(f"⚠ Could not kill process on port {port}: {e}")
        return False

def find_available_port(start_port=5000, max_attempts=10):
    """Find an available port starting from start_port"""
    for port in range(start_port, start_port + max_attempts):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('', port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"Could not find an available port in range {start_port}-{start_port+max_attempts}")

# Stop existing server if running
if 'server' in globals(): 
    try:
        server.stop()
        time.sleep(0.5)
    except:
        pass

# Try to kill any process on preferred port, then find available port
preferred_port = 7020
kill_process_on_port(preferred_port)
port = find_available_port(preferred_port)

app = FastHTML(hdrs=Theme.blue.headers(title="Page Snippets", mode="dark"))
rt = app.route

try:
    server = JupyUvi(app, port=port)
    preview = partial(HTMX, app=app, port=port)
    print(f"✓ Server running on port {port}")
except Exception as e:
    print(f"✗ Failed to start server: {e}")
    raise

✓ Server running on port 7020


## Hero Section

Hero sections for landing pages with CTA buttons.

In [ ]:
#| export
def HeroSection(
    title: str,              # Main hero title
    subtitle: str,           # Hero subtitle/description
    primary_cta_text: str,   # Primary CTA button text
    primary_cta_href: str,   # Primary CTA button href
    secondary_cta_text: str = None,  # Secondary CTA button text (optional)
    secondary_cta_href: str = None,  # Secondary CTA button href (optional)
    background: str = "bg-primary-container",  # BeerCSS background class
    cls: str = "",  # Additional classes
    style: str | None = None,  # Optional inline style (avoid by default)
    extra_css: str | None = None,  # Optional custom CSS (minimal; opt-in)
    extra_js: str | None = None,  # Optional custom JS (minimal; opt-in)
 ):
    """Hero section (BeerCSS-first).
    
    Note: Avoid `middle-align` on the outer stack; in BeerCSS it can turn the
    wrapper into a horizontal flex row, which makes the H1/P/CTA sit side-by-side.
    """
    hooks = []
    if extra_css:
        hooks.append(Style(extra_css))
    if extra_js:
        hooks.append(Script(extra_js))
    
    ctas = [Button(primary_cta_text, href=primary_cta_href, variant="raised")]
    if secondary_cta_text and secondary_cta_href:
        ctas.append(Button(secondary_cta_text, href=secondary_cta_href, variant="outlined"))
    
    stack = DivVStacked(
        H1(title, cls="no-margin"),
        P(subtitle, cls="secondary-text"),
        Div(*ctas, cls="row middle-align center-align small-space"),
        cls="center-align large-padding",
    )
    content = Div(stack, cls="responsive")
    return Section(
        *hooks,
        content,
        cls=f"{background} {cls}".strip(),
        style=style,
    )

In [ ]:
#| code-fold: true
#| eval: false

def hero_section():
    return HeroSection(
    title="Welcome to Material UI",
    subtitle="Build beautiful web apps with FastHTML",
    primary_cta_text="Get Started",
    primary_cta_href="/signup",
    secondary_cta_text="Learn More",
    secondary_cta_href="/docs"
) 

preview(hero_section())

## Features Grid

Grid of feature cards with icons.

In [ ]:
#| export
def FeaturesGrid(
    features: list,  # List of dicts with 'icon', 'title', 'description'
    title: str = None,  # Optional section title
    subtitle: str = None,  # Optional section subtitle
    cols: int = 3,  # Number of columns (1-4)
    cls: str = ""  # Additional classes
):
    """Grid of feature cards using BeerCSS grid helpers."""
    import fh_matui.components as _cmp
    
    GridCell = getattr(_cmp, 'GridCell', None)
    ResponsiveGrid = getattr(_cmp, 'ResponsiveGrid', None)
    
    if GridCell is None or ResponsiveGrid is None:
        # Fallback: keep previews working even if the notebook kernel has a stale module import.
        def GridCell(*c, span=(), cls='', **kwargs):
            cell_cls = []
            cell_cls.extend(normalize_tokens(span))
            cell_cls.extend(normalize_tokens(cls))
            cell_cls = [t for t in cell_cls if t]
            return Div(*c, cls=stringify(dedupe_preserve_order(cell_cls)), **kwargs)
        
        def ResponsiveGrid(*cells, space='medium-space', cls: str = '', **kwargs):
            cls_tokens = normalize_tokens(cls)
            grid_cls = ['grid']
            if space and space not in cls_tokens:
                grid_cls.extend(normalize_tokens(space))
            grid_cls.extend(cls_tokens)
            grid_cls = [t for t in grid_cls if t]
            return Div(*cells, cls=stringify(dedupe_preserve_order(grid_cls)), **kwargs)
    
    cols = max(1, min(int(cols), 4))
    col_width = 12 // cols  # 12-col grid
    
    s_span = "s12"
    m_span = "m12" if cols == 1 else "m6"
    l_span = f"l{col_width}"
    cell_span = f"{s_span} {m_span} {l_span}".strip()
    
    cells = [
        GridCell(
            Card(
                Div(Icon(f["icon"], size="medium", cls="primary-text"), cls="center-align"),
                H5(f["title"], cls="center-align small-margin"),
                P(f["description"], cls="secondary-text center-align"),
                cls="center-align padding",
            ),
            span=cell_span,
        )
        for f in features
    ]
    
    content = []
    if title:
        content.append(H1(title, cls="center-align bottom-margin"))
    if subtitle:
        content.append(P(subtitle, cls="center-align secondary-text large-margin large-text"))
    
    content.append(ResponsiveGrid(*cells, space="medium-space"))
    return Section(*content, cls=f"padding horizontal-padding {cls}".strip())

In [ ]:
#| code-fold: true
#| eval: false

sample_features = [
    {'icon': 'star', 'title': 'Fast', 'description': 'Lightning fast performance'},
    {'icon': 'shield', 'title': 'Secure', 'description': 'Bank-level security'},
    {'icon': 'trending_up', 'title': 'Scalable', 'description': 'Grows with your needs'},
        {'icon': 'star', 'title': 'Fast', 'description': 'Lightning fast performance'},
    {'icon': 'shield', 'title': 'Secure', 'description': 'Bank-level security'},
    {'icon': 'trending_up', 'title': 'Scalable', 'description': 'Grows with your needs'},
]

@rt('/test-fg')
def test_fg():
    return FeaturesGrid(
        features=sample_features,
        title="Why Choose Us",
        subtitle="Everything you need to ship quickly",
        cols=3,
    )

In [ ]:
#| code-fold: true
#| eval: false

preview(test_fg())

## Pricing Section

Simple pricing section: one card with a checklist and CTA.

In [ ]:
#| export

def PricingSection(

    title: str,
    price: str,
    plan_name: str,
    features: list,
    cta_text: str,
    cta_href: str,
    period: str = "month",
    cls: str = "",
):
    """Simple pricing card: title, bullets, button."""
    
    # Build feature list items
    feature_items = []
    for feature in features:
        feature_items.append(
            Li(
                Icon("check", cls="small primary-text"),
                Span(feature, cls="small-space"),
                cls="row middle-align small-space",
            )
        )
    
    # Build the card
    card = Card(
        # Header
        H3(plan_name, cls="center-align"),
        H2(price, cls="center-align no-margin"),
        P(f"per {period}", cls="center-align secondary-text"),
        
        # Features
        Ul(*feature_items, cls="no-margin"),
        
        # CTA
        Button(cta_text, href=cta_href, cls="responsive large-margin"),
        
        cls="medium-width padding round",
    )
    
    # Center the card on the page
    return Section(
        H2(title, cls="center-align"),
        Div(card, cls="center-align"),
        cls=f"padding {cls}".strip(),
    )

In [ ]:
#| code-fold: true
#| eval: false

pricing = PricingSection(
        title="Simple Pricing",
        price="$7.99",
        plan_name="Professional",
        features=[
            "Unlimited users",
            "24/7 priority support",
            "Custom branding options",
            "Advanced analytics dashboard",
            "Full API access",
            "Priority request queue",
        ],
        cta_text="Get Started",
        cta_href="/signup",
        period="month",
    )

preview(pricing())


## FAQ Section

FAQ section using native Details/Summary elements.

In [ ]:
#| export
def FAQSection(title: str, faqs: list, cls: str = ""):
    """FAQ section (BeerCSS-first).

    Goal: add a *small* vertical space between each question item (between <details> blocks).
    """
    items = [
        Div(
            Details(
                Summary(
                    Article(
                        Nav(Div(faq['question'], cls="max bold"), I("expand_more")),
                        cls="round primary no-elevate",
                    )
                ),
                Article(P(faq['answer'], cls="secondary-text"), cls="round border padding"),
            ),
            # Small visual gap between questions.
            cls="small-margin",
        )
        for faq in faqs
    ]
    
    return Section(
        H2(title, cls="center-align bottom-margin"),
        Div(*items, cls="column"),
        cls=f"responsive column {cls}".strip(),
    )

In [ ]:
#| code-fold: true
#| eval: false

faqs = [
    {'question': 'What is FastHTML?', 'answer': 'FastHTML is a modern web framework'},
    {'question': 'How much does it cost?', 'answer': 'See our pricing page for details'}
]

def faq_section():
    return  FAQSection(
    title="Frequently Asked Questions",
    faqs=faqs
)

preview(faq_section())

## Footer

Footer with multiple columns, social links, and copyright.

In [ ]:
#| export
def PageFooter(
    columns: list,         # List of dicts with 'title' and 'links' (list of dicts with 'text' and 'href')
    copyright: str,        # Copyright text
    social_links: list = None,  # List of dicts with 'icon' and 'href'
    logo: str = None,      # Optional logo text
    cls: str = ""          # Additional classes
):
    """Footer with multiple columns, social links, and copyright"""
    footer_cols = []
    
    # Logo column if provided
    if logo:
        footer_cols.append(
            Div(
                H6(logo, cls='no-margin bold'),
                cls='padding'
            )
        )
    
    # Link columns
    for col in columns:
        links = [A(link['text'], href=link['href'], cls='grey-text') for link in col.get('links', [])]
        footer_cols.append(
            Div(
                H6(col['title'], cls='no-margin bold'),
                Div(*links, cls='column small-space'),
                cls='padding'
            )
        )
    
    # Footer row with full spacing between columns
    footer_row = DivFullySpaced(*footer_cols)
    
    # Bottom section with copyright and social links
    bottom_left = Div(P(copyright, cls='small-text grey-text no-margin'))
    
    bottom_right = Div()
    if social_links:
        social_icons = [A(Icon(s['icon']), href=s['href'], cls='grey-text') for s in social_links]
        bottom_right = Div(*social_icons, cls='row small-space')
    
    bottom_section = DivFullySpaced(bottom_left, bottom_right)
    
    footer_cls = f'surface-container padding {cls}'.strip()
    return ft_hx('footer')(
        Div(
            footer_row,
            Hr(),
            bottom_section,
            cls=f"responsive {footer_cls}"
        )
    )


In [ ]:
#| code-fold: true
#| eval: false

def ex_footer():
    return   PageFooter(
    columns=[
        {'title': 'Product', 'links': [{'text': 'Features', 'href': '#features'}]},
        {'title': 'Company', 'links': [{'text': 'About', 'href': '/about'}]}
    ],
    copyright="© 2025 MyBrand",
    social_links=[{'icon': 'twitter', 'href': 'https://twitter.com'}],
    logo="MyBrand"
)

preview(ex_footer())

In [ ]:
#| export
def LandingNavBar(
    brand_name: str,        # Brand name for the navbar
    links: list = None,     # List of dicts with 'text' and 'href'
    actions: list = None,   # List of action buttons (Buttons/Links)
    sticky: bool = True,    # Whether navbar sticks to top
    cls: str = ""           # Additional classes
):
    """
    Landing page navigation bar using Toolbar component with brand and links.
    
    Args:
        brand_name: Brand name/logo text
        links: List of navigation links [{'text': 'Features', 'href': '#features'}, ...]
        actions: List of action elements (Buttons, etc.) for CTA
        sticky: Whether navbar sticks to top while scrolling (default: True)
        cls: Additional CSS classes
    
    Example:
        LandingNavBar(
            brand_name="MyBrand",
            links=[
                {'text': 'Features', 'href': '#features'},
                {'text': 'Pricing', 'href': '#pricing'},
                {'text': 'FAQ', 'href': '#faq'}
            ],
            actions=[
                Button("Sign Up", href="/signup", variant="raised")
            ]
        )
    """
    # Build navigation elements
    nav_items = []
    
    # Add brand
    nav_items.append(H5(brand_name, cls="no-margin bold"))
    
    # Add spacer to push links to the right
    nav_items.append(Div(cls="max"))
    
    # Add navigation links
    if links:
        for link in links:
            nav_items.append(
                A(link['text'], href=link['href'], cls="padding")
            )
    
    # Add action buttons
    if actions:
        nav_items.extend(actions)
    
    # Use Toolbar with sticky positioning and blur effect
    toolbar_cls = f"blur {'sticky top' if sticky else ''} {cls}".strip()
    
    return Toolbar(Container(*nav_items, responsive=True), cls=toolbar_cls, fill=True, elevate='medium')

## Landing Page

Complete landing page combining hero, features, pricing, FAQs, and footer.

In [ ]:
#| code-fold: true
#| eval: false

def LandingPage(
    brand_name: str,        # Brand name for navbar
    hero_title: str,        # Hero section title
    hero_subtitle: str,     # Hero section subtitle
    features: list,         # Features list
    faqs: list,             # FAQ list
    demo_modal_id: str = "demo-modal",  # Kept for backward-compat; not used by default
    cls: str = ""           # Additional classes
):
    """Complete landing page (BeerCSS-first)."""
    nav = LandingNavBar(
        brand_name=brand_name,
        links=[
            {'text': 'Features', 'href': '#features'},
            {'text': 'Pricing', 'href': '#pricing'},
            {'text': 'FAQ', 'href': '#faq'},
        ],
        actions=[
            Button("Sign Up", href="/signup", variant="raised"),
        ],
    )
    hero = HeroSection(
        title=hero_title,
        subtitle=hero_subtitle,
        primary_cta_text="Get Started",
        primary_cta_href="/signup",
        secondary_cta_text="Learn More",
        secondary_cta_href="#features",
        background="bg-primary-container",
    )
    pricing = PricingSection(
        title="Simple Pricing",
        price="$7.99",
        plan_name="Professional",
        features=[
            "Unlimited users",
            "24/7 priority support",
            "Custom branding options",
            "Advanced analytics dashboard",
            "Full API access",
            "Priority request queue",
        ],
        cta_text="Get Started",
        cta_href="/signup",
        period="month",
    )
    return Div(
        nav,
        hero,
        Div(
            Div(id="features")(
                FeaturesGrid(
                    features=features,
                    title="Why Choose Us",
                    subtitle="Everything you need to build amazing web apps",
                    cols=3,
                )
            ),
            Div(id="pricing")(pricing),
            Div(id="faq")(FAQSection(title="Frequently Asked Questions", faqs=faqs)),
            cls="responsive column large-space",
        ),
        PageFooter(
            columns=[
                {'title': 'Product', 'links': [
                    {'text': 'Features', 'href': '#features'},
                    {'text': 'Pricing', 'href': '#pricing'},
                ]},
                {'title': 'Company', 'links': [
                    {'text': 'About', 'href': '/about'},
                    {'text': 'Blog', 'href': '/blog'},
                    {'text': 'Contact', 'href': '/contact'},
                ]},
                {'title': 'Legal', 'links': [
                    {'text': 'Privacy', 'href': '/privacy'},
                    {'text': 'Terms', 'href': '/terms'},
                ]},
            ],
            copyright=f"© 2025 {brand_name}. All rights reserved.",
            social_links=[
                {'icon': 'twitter', 'href': 'https://twitter.com'},
                {'icon': 'github', 'href': 'https://github.com'},
                {'icon': 'linkedin', 'href': 'https://linkedin.com'},
            ],
            logo=brand_name,
        ),
        cls=f"column {cls}".strip(),
    )

In [ ]:
#| code-fold: true
#| eval: false

def LandingPageV2_nav(
    brand_name: str,
    links: list | None = None,
    cls: str = "",
) -> FastHTML:
    """Landing page V2 demo (BeerCSS-first).

    Focus: all content sits inside a centered container (not full width).
    """
    nav_links = links or [
        {"text": "Overview", "href": "#overview"},
        {"text": "Features", "href": "#features"},
        {"text": "Pricing", "href": "#pricing"},
    ]

    link_items = [A(item["text"], href=item["href"], cls="padding") for item in nav_links]

    toolbar = Toolbar(
        Div(
            H5(brand_name, cls="no-margin bold"),
            Div(cls="max"),
            *link_items,
            cls="row wrap middle-align fully-spaced",
        ),
        cls="round surface-container-highest blur shadow-medium",
        fill=True,
        elevate="medium",
    )

    hero_section = HeroSection(
        title="Build faster with FastHTML",
        subtitle="Compose production-ready pages with Material inspired components wired for HTMX.",
        primary_cta_text="Get Started",
        primary_cta_href="/signup",
        secondary_cta_text="View Docs",
        secondary_cta_href="/docs",
        background="bg-primary-container",
        cls="round shadow-medium",
    )

    grid_features = [
        {"icon": "dashboard", "title": "Page primitives", "description": "Hero, feature, pricing, and FAQ sections tuned for Material theming."},
        {"icon": "bolt", "title": "HTMX ready", "description": "Each preview runs live inside the notebook server for instant validation."},
        {"icon": "palette", "title": "Color tokens", "description": "Swap Material container classes to theme individual sections quickly."},
        {"icon": "extension", "title": "Composable APIs", "description": "All helpers accept extra classes so you can extend without forking."},
        {"icon": "security", "title": "Accessible defaults", "description": "Typography and spacing follow WCAG friendly defaults out of the box."},
        {"icon": "rocket_launch", "title": "Fast iteration", "description": "nbdev keeps code and docs together with automatic exports."},
    ]

    feature_grid_section = FeaturesGrid(
        features=grid_features,
        title="All the sections you need",
        subtitle="Mix heroes, feature highlights, grids, pricing, and FAQs without leaving the design system.",
        cols=3,
        cls="round shadow-medium large-padding",
    )

    faq_entries = [
        {"question": "Can I mix components?", "answer": "Yes. Every helper returns plain FastHTML nodes so you can reorder or nest them."},
        {"question": "How do colors work?", "answer": "Apply Material container classes such as bg-primary-container or surface-container-high to any wrapper."},
        {"question": "Is HTMX required?", "answer": "No. The interactions degrade gracefully, but HTMX hooks are available when you need them."},
    ]

    faq_section = FAQSection(
        title="Frequently asked questions",
        faqs=faq_entries,
        cls="round shadow-medium",
    )

    footer = PageFooter(
        columns=[
            {"title": "Product", "links": [{"text": "Components", "href": "#features"}, {"text": "Guides", "href": "/guides"}]},
            {"title": "Resources", "links": [{"text": "Docs", "href": "/docs"}, {"text": "Playground", "href": "/play"}]},
            {"title": "Company", "links": [{"text": "About", "href": "/about"}, {"text": "Contact", "href": "/contact"}]},
        ],
        copyright=f"(c) 2026 {brand_name}. All rights reserved.",
        social_links=[
            {"icon": "github", "href": "https://github.com"},
            {"icon": "twitter", "href": "https://twitter.com"},
            {"icon": "slack", "href": "https://slack.com"},
        ],
        logo=brand_name,
        cls="round shadow-medium large-padding",
    )

    toolbar_block = Div(toolbar, cls="row center-align bottom-margin")

    hero_block = Section(
        H2("Hero", cls="no-margin"),
        P("Primary pitch area with generous spacing.", cls="secondary-text"),
        hero_section,
        cls="round bg-primary-container shadow-medium column gap-medium large-padding",
    )

   

    feature_grid_block = Section(
        H2("Feature grid", cls="no-margin"),
        P("Icon cards for quick scanning.", cls="secondary-text"),
        feature_grid_section,
        cls="round bg-tertiary-container shadow-medium column gap-medium large-padding",
    )

    faq_block = Section(
        H2("FAQs", cls="no-margin"),
        P("Support answers in collapsible cards.", cls="secondary-text"),
        faq_section,
        cls="round surface-container-high shadow-medium column gap-medium large-padding",
    )

    footer_block = Section(
        H2("Footer", cls="no-margin"),
        P("Wrap up with key links and social proof.", cls="secondary-text"),
        footer,
        cls="round surface-container shadow-medium column gap-medium large-padding",
    )

    # Put layout classes on an inner wrapper; `Container(...)` provides centered width constraint.
    inner = Div(
        toolbar_block,
        hero_block,
      
        feature_grid_block,
        faq_block,
        footer_block,
        cls="responsive column gap-xlarge",
    )

    return Section(
        Container(inner),
        cls=f"bg-primary-container large-padding {cls}".strip(),
    )

In [ ]:
#| code-fold: true
#| eval: false

def landing_page_v2_nav():
    return LandingPageV2_nav(brand_name="TestBrand")

@rt('/test-lp')
def test_landing_page_v2_nav():
    return landing_page_v2_nav()

In [ ]:
#| hide

import nbdev as nb
nb.nbdev_export()